Name :  Nitesh Ramesh Morem  ,  Kashif Riyaz 

Date : ***20/07/2025***

Email : ***n.morem@oth-aw.de*** , ***k.kashif-riyaz@oth-aw.de***

Dep  :  ***MAI***

To understand the workFlow please folow this link to out Miro Board which shows our workFlow

https://miro.com/app/board/uXjVIgh2pSA=/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df= pd.read_csv('imu.csv')  # this is the IMU DATA
df_2= pd.read_csv('rssi.csv')  # This is the Rssi data
 

#  Compute Magnetometer-Based Heading (`theta_mag`)

This code calculates the compass heading (yaw angle) from magnetometer readings. It uses the horizontal magnetic field components (`mag_x` and `mag_y`) to estimate orientation relative to magnetic North.


In [ ]:
import numpy as np

#  Compute heading in radians using magnetometer x and y axes
# arctan2 handles full 360° quadrant-aware angle calculation
heading_rad = np.arctan2(df['mag_y'], df['mag_x'])

#  Convert the heading from radians to degrees
heading_deg = np.degrees(heading_rad)

# Normalize the heading to the range [0, 360) degrees
heading_deg = (heading_deg + 360) % 360

#  Store the result in the DataFrame
# This gives the magnetometer-based heading (in degrees)
df['theta_mag'] = heading_deg


##  Tilt-Compensated Heading Calculation (`theta_tilt_comp`)

This section computes the compass heading from magnetometer data, corrected for device tilt using accelerometer data. This helps improve accuracy when the device is not lying flat.



In [ ]:

#  Compute pitch and roll from accelerometer data
# These represent the tilt of the device in space
pitch = np.arcsin(-df['acc_x'] / np.sqrt(df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2))
roll = np.arcsin(df['acc_y'] / np.sqrt(df['acc_y']**2 + df['acc_z']**2))

#      Compensate magnetometer readings for tilt
# These formulas correct the magnetic field values to horizontal plane
Xh = df['mag_x'] * np.cos(pitch) + df['mag_z'] * np.sin(pitch)
Yh = (df['mag_x'] * np.sin(roll) * np.sin(pitch) +
      df['mag_y'] * np.cos(roll) -
      df['mag_z'] * np.sin(roll) * np.cos(pitch))

#  Calculate tilt-compensated heading
heading_rad_tc = np.arctan2(Yh, Xh)
heading_deg_tc = (np.degrees(heading_rad_tc) + 360) % 360  # Normalize to [0, 360)

# Step 4: Save result to DataFrame
df['theta_tilt_comp'] = heading_deg_tc

In [ ]:
# visulaising the  raw heading and tilt compensated
plt.figure(figsize=(20, 5))
plt.plot(df['theta_mag'], label='Raw Heading')
plt.plot(df['theta_tilt_comp'], label='Tilt Compensated Heading', alpha=0.7)
plt.legend()
plt.title("Heading from Magnetometer")
plt.ylabel("Degrees")
plt.xlabel("Time Index")
plt.grid(True)
plt.show()


TO understand the workFlow please folow this link to out Miro Board which shows our workFlow

https://miro.com/app/board/uXjVIgh2pSA=/

In [ ]:

# Compute heading from magnetometer
# Use horizontal components of the magnetic field to estimate yaw angle
heading_rad = np.arctan2(df['mag_y'], df['mag_x'])         # Handles quadrant correctly
heading_deg = np.degrees(heading_rad)                      # Convert radians to degrees
heading_deg = (heading_deg + 360) % 360                    # Normalize to [0, 360)
df['theta_mag'] = heading_deg                              # Store as theta_mag

# here we are Configure complementary filter parameters
alpha = 0.98           # Trust gyro 98%, magnetometer 2%
dt = 1 / 50.0          # Sampling interval in seconds (e.g., 50Hz sensor)

#  we Integrate gyroscope data (gyro_z) to estimate angle change over time
gyro_z = df['gyro_z'].fillna(0).to_numpy()                # Fill NaNs for safe computation
theta_gyro = np.zeros_like(gyro_z)
for i in range(1, len(gyro_z)):
    theta_gyro[i] = theta_gyro[i-1] + gyro_z[i] * dt      # Cumulative integration

# Convert integrated gyro angle from radians to degrees for better readibility
theta_gyro_deg = np.degrees(theta_gyro)

#  Apply complementary filter to combine gyro and magnetometer headings
theta_fused = np.zeros_like(theta_gyro_deg)
theta_fused[0] = df['theta_mag'].iloc[0]                  # we then Initialize with magnetometer heading

for i in range(1, len(theta_fused)):
    gyro_term = theta_fused[i-1] + gyro_z[i] * dt * 180 / np.pi   # Gyro contribution (deg)
    mag_term = df['theta_mag'].iloc[i]                             # Magnetometer contribution
    theta_fused[i] = alpha * gyro_term + (1 - alpha) * mag_term   # Weighted average so that we can easily control the heading

# then we Normalize the fused heading to [0, 360)
theta_fused = (theta_fused + 360) % 360

# now finally Store fused heading in the DataFrame
df['theta_fused'] = theta_fused

In [ ]:
 

# Smooth the fused heading for better visualization
df['theta_fused_smoothed'] = df['theta_fused'].rolling(window=100, min_periods=1).mean()

# Plot the smoothed fused heading and raw magnetometer heading
plt.figure(figsize=(20, 5))
plt.plot(df['theta_fused_smoothed'], label='Smoothed Fused Heading (theta_fused)', linewidth=2)
plt.plot(df['theta_mag'], label='Raw Magnetometer Heading (theta_mag)', alpha=0.5)

plt.title("Fused Heading (Gyro + Magnetometer)")
plt.xlabel("Sample Index")
plt.ylabel("Heading (Degrees)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


##   Visualization: Fused Heading vs Raw Magnetometer Heading

This plot compares the raw magnetometer-based heading (`theta_mag`) with the sensor-fused heading (`theta_fused`) that combines gyroscope and magnetometer data using a complementary filter.

- The orange line shows the raw magnetometer heading, which is noisy and can spike due to local magnetic disturbances which not really reliable for us .
- The blue line shows the smoothed and fused heading using the complementary filter which we implemented anove, which is more stable and robust against noise  but still it is not realiable as we need better heading.

some Key Benefits of Sensor Fusion:
- Smooths out jitter and spikes in magnetometer readings.
- Maintains responsiveness to rapid heading changes using gyroscope input.
- Provides a cleaner estimate of orientation for motion tracking or navigation applications.





>> but in our case we need to more accurate heading as the turns are very noisy. this is due to the reason the sampling rate for magnetometer is 20hz and the sampling rate for acc and gyro is 50hz which makes the magnetometer very noisy and previous samples are attached untill a new value is recevied with arduino.
